# 🌐 06-分布式

> 面试定位：**CAP / 一致性 / 消息队列 / 分布式事务 / 限流熔断** 是后端与算法岗的必考高频。本文件按"理论 → 中间件 → 实践方案"组织，每节含原理、对比表、可运行代码与面试追问。
> 标记：🔴 必背 / 🟡 掌握 / 🟢 了解。

---

> 面试定位：**CAP / 一致性 / 消息队列 / 分布式事务 / 限流熔断** 是后端与算法岗必考高频。
> 标记：🔴 必背 / 🟡 掌握 / 🟢 了解。本 notebook 配套 `06-分布式.md` 速查文档，全部代码可运行。


## §1 CAP 理论（🔴 必背）


![dist_cap.png](images/dist_cap.png)

*配图来源：本仓库 matplotlib 绘制（`images/dist_cap.png`）*


### 1.1 三个要素的定义

- **C（Consistency 一致性）**：所有节点在同一时刻读到**相同的数据**（线性一致性：任何读都能返回最近一次写的值）。
- **A（Availability 可用性）**：只要收到请求，**每个请求都必须成功响应**（不保证返回最新数据，但必须"活着能答"）。
- **P（Partition tolerance 分区容错）**：网络发生分区（节点间消息丢失/延迟）时，系统**仍能继续运行**。

> ⚠️ 精确表述：CAP 不是"三选二"，而是"**当发生网络分区时，只能从 C 和 A 中选一个**"。没有分区时 C 和 A 可以同时满足（如单机数据库）。


### 1.2 为什么"只能三选二"（关键论证）

假设一个分布式系统有节点 **N1、N2**，客户端 **C** 先向 N1 写入了值 `v1`，随后发生网络分区（N1 与 N2 之间断开）：

| 选择 | 行为 | 后果 |
|------|------|------|
| 选 **CP** | 分区期间 N2 拒绝读请求（返回错误） | 牺牲可用性，保证一致性 |
| 选 **AP** | 分区期间 N2 用旧数据应答（返回 `v0`） | 牺牲一致性，保证可用性 |
| 选 **CA** | 不可能：分区已经发生，想同时一致又可用无解 | — |

**核心一句话**：网络分区是**必然会发生**的现实（不是假设），所以在设计时只能二选一。


### 1.3 实际系统的 CP/AP 归类

| 系统 | 取舍 | 原因 | 典型场景 |
|------|------|------|---------|
| **ZooKeeper** | CP | 选举/过半提交，leader 失联时集群不可写 | 分布式协调、元数据 |
| **etcd** | CP | Raft 实现强一致 | k8s 元数据 |
| **Eureka** | AP | 节点间互相注册，失效节点不摘除仍响应 | 服务发现（宁可错不可停） |
| **Nacos** | 默认 AP，可切 CP | 支持两种模型 | 服务发现/配置 |
| **Redis Cluster** | AP（主从同步异步） | 主挂了从可能丢数据 | 缓存（容忍少量丢失） |
| **MySQL 主从** | 弱一致（最终一致） | binlog 异步复制 | 读写分离 |


### 1.4 面试追问

- **问**："为什么说 CAP 是伪命题？"
  - **答**：因为分区是现实，所以系统必须保证 P，实际是 **CP vs AP** 的选择题；且 CAP 只描述"分区发生那一瞬"，恢复后 C 和 A 都会回来。
- **问**："你设计系统选 CP 还是 AP？"
  - **答**：看业务。**钱、库存、订单**选 CP（宁可拒服务不可错账）；**feed、点赞数、在线状态**选 AP（可用性优先）。并补一句"加缓存/异步补偿可以兼顾"。

---


## §2 一致性模型（🟡 掌握）


### 2.1 强弱一致全谱

| 模型 | 描述 | 实现 | 例子 |
|------|------|------|------|
| **强一致**（线性一致性） | 读总能看到最近写 | Raft/Paxos 过半提交 | ZooKeeper、etcd |
| **顺序一致** | 所有进程看到相同操作顺序 | 全局序 | 单机内存模型 |
| **因果一致** | 有因果关系的操作有序，并发可乱 | 版本向量 | 聊天系统 |
| **最终一致** | 无新写后，经过一段时间各副本收敛 | 异步复制/冲突解决 | DNS、Git、社交 |

**收敛时间（不一致窗口）**取决于复制延迟：MySQL 主从几毫秒~秒级，CDN 秒级~分钟级。


### 2.2 Raft 核心思想（🔴 背 3 个点）

Raft 把共识拆成 3 个子问题，可读性远超 Paxos：

1. **Leader 选举（Election）**：节点有角色 `Leader / Follower / Candidate`。Follower 超时未收到心跳 → 自增 term 并投票；**过半票**当选。任期 term 单调递增，防止旧 Leader 分裂。
2. **日志复制（Log replication）**：客户端请求只发给 Leader，Leader 追加到本地日志 → 广播给 Follower → **过半节点确认后提交（commit）**并应用到状态机。
3. **安全性（Safety）**：选举时只选"日志最新"的候选者（比较 lastLogIndex/lastLogTerm）；已提交的日志不会被覆盖——**保证已提交日志永不丢失**。


### 2.3 Paxos vs Raft 对比

| 维度 | Paxos | Raft |
|------|-------|------|
| 理解难度 | 高（多轮提案、多数派含义难懂） | 低（状态机式角色切换） |
| Leader | 无固定 Leader（Basic Paxos） | 有明确 Leader |
| 日志顺序 | 不保证连续 | 保证连续（日志空洞用 prevLogTerm 校验） |
| 实现 | 复杂、易出错 | 工业界事实标准（etcd/consul/tiKV） |
| 成员变更 | 难 | Joint Consensus 解决 |


### 2.4 面试追问

- **问**："Raft 如何保证不丢已提交日志？"
  - **答**：① 过半确认才提交；② 选举限制（候选者日志必须不比当前 Leader 旧）；③ 新 Leader 通过 prevLogIndex/prevLogTerm 强制覆盖不一致日志，但**只覆盖未提交的**。
- **问**："最终一致怎么检测数据冲突？"
  - **答**：版本号/时间戳/向量时钟，冲突时按业务策略合并（last-write-wins、CRDT、人工仲裁）。

---


## §3 消息队列（🔴 必背）


### 3.1 三大作用与一个代价

| 作用 | 说明 | 示例 |
|------|------|------|
| **异步** | 主流程不等待耗时不重要的子任务 | 下单 → 发短信/邮件 |
| **削峰** | 请求先入队，下游按自身节奏消费 | 秒杀、双 11 流量洪峰 |
| **解耦** | 生产/消费方互不知晓，可独立扩缩容 | 订单服务 → 库存/积分/日志 |

**代价**：多一跳 → 延迟增加、消息可能丢/重复、需保证顺序时更复杂。**能不用就不用，用了就要想清楚三件事**。


### 3.2 如何保证不丢消息（三环节逐一击破）

| 环节 | 风险 | 对策 |
|------|------|------|
| **生产端** | 发送失败/发送成功但没落盘 | `send()` 同步确认（ack）；失败重试 + 本地消息表兜底 |
| **Broker 存储** | 机器宕机内存数据丢失 | 刷盘策略（Kafka `acks=all` + `min.insync.replicas`）；副本机制（ISR） |
| **消费端** | 消费后未提交 offset 就 crash → 重复消费 | **先处理业务再提交 offset**（at-least-once）；幂等兜底 |

> ⚠️ Kafka 默认是 **at-least-once**（至少一次）：不丢但可能重复 → 必须配合**幂等消费**。要 exactly-once 需事务 + 幂等 Producer（较复杂）。


### 3.3 如何保证不重复消费（幂等三招）

1. **唯一键去重表**：消费前 INSERT IGNORE 业务唯一键（订单号/流水号），插入失败说明已处理。
2. **版本号/状态机**：`UPDATE ... WHERE version = ?`，影响行数为 0 表示已被处理。
3. **Redis SetNX**：`SETNX biz:order:123 1`，返回 0 表示已消费过。


### 3.4 如何保证顺序消息

- **单分区模型**：同一业务 key 哈希到**同一个分区**，分区内天然有序（Kafka）；消费时单线程。
- **RocketMQ**：`MessageQueueSelector` 按订单 ID 选队列，同一订单进同一队列。
- **RabbitMQ**：单队列 + 单消费者（牺牲并行度换顺序）。


### 3.5 主流 MQ 对比（🟡 掌握）

| 特性 | Kafka | RocketMQ | RabbitMQ | Pulsar |
|------|-------|----------|----------|--------|
| 模型 | Topic/Partition | Topic/Queue | Exchange/Queue | Topic/Subscription |
| 吞吐 | 极高（百万级/s） | 高（十万级） | 中（万级） | 极高 |
| 顺序 | 分区内有序 | 队列内有序 | 单队列有序 | 订阅内有序 |
| 延迟 | 毫秒级 | 毫秒级 | 微秒级 | 毫秒级 |
| 消费确认 | offset 手动/自动 | ack | ack | ack |
| 适用 | 日志、大数据、流处理 | 电商事务消息、顺序 | 轻量业务解耦 | 云原生多租户 |


### 3.6 面试追问

- **问**："Kafka 为什么快？"
  - **答**：顺序写磁盘（追加日志）+ 页缓存（读命中内存）+ 零拷贝（sendfile 直接把页缓存发到网卡）+ 批量压缩。
- **问**："消息积压了怎么办？"
  - **答**：先扩容消费者（超过分区数需加分区），紧急时可临时转储（写文件/Redis），再逐步消费；定位是消费慢还是生产快。

---


## §4 分布式事务（🟡 掌握，方案对比必考）


### 4.1 2PC（两阶段提交）

**流程**：
1. **Prepare（阶段一）**：协调者向所有参与者发 prepare，参与者执行事务但**不提交**，本地写 undo/redo 日志，返回"可提交/不可提交"。
2. **Commit/Abort（阶段二）**：全部同意 → 广播 commit；任一反对/超时 → 广播 rollback。

**三大问题**：
- **同步阻塞**：Prepare 后参与者一直占资源等待协调者指令。
- **协调者单点**：协调者宕机则全体卡死。
- **脑裂风险**：阶段二协调者与部分参与者失联，网络恢复后状态不一致。

**改进**：3PC 引入超时与 preCommit 减少阻塞，但仍不是严格共识；业界多用 **Paxos/Raft 替换协调者**（如 MySQL Group Replication）。


### 4.2 TCC（Try-Confirm-Cancel）

| 阶段 | 动作 | 例子（扣款+减库存） |
|------|------|---------------------|
| **Try** | 预留资源（不真正扣） | 冻结 100 元、锁定 1 件库存 |
| **Confirm** | 确认执行（Try 全部成功） | 真正扣款、真正减库存 |
| **Cancel** | 回滚（任一 Try 失败） | 解冻金额、释放库存 |

**优点**：业务可控、性能好（无长事务锁）；**缺点**：**侵入性强**，每个业务要写 Try/Confirm/Cancel 三套逻辑；要保证 Confirm/Cancel 幂等。


### 4.3 Saga / 本地消息表（最终一致方案）

**Saga**：一个长事务拆成多个本地事务，每个都有**正向操作 + 补偿操作**；失败时逆序执行补偿。
- 编排式（Orchestration）：中央 Saga 引擎调度。
- 协同式（Choreography）：事件驱动链式触发。

**本地消息表**：业务表和消息表**同一个本地事务**写入；后台任务扫描消息表发 MQ，消费成功后删记录——**生产端不丢消息的最简单可靠方案**。


### 4.4 方案对比表（面试直接背）

| 方案 | 一致性 | 性能 | 业务侵入 | 适用 |
|------|--------|------|----------|------|
| 2PC | 强 | 低（长锁） | 低 | 单机强一致、短事务 |
| 3PC | 较强 | 中 | 低 | 减少阻塞的场景 |
| TCC | 强（业务保证） | 中 | **高**（三套代码） | 资金类、必须实时确认 |
| Saga | 最终一致 | 高 | 中（补偿逻辑） | 长流程、允许短暂不一致 |
| 本地消息表 | 最终一致 | 高 | 中 | 与 MQ 配合的生产端可靠性 |
| MQ 事务消息 | 最终一致 | 高 | 低（半消息机制） | RocketMQ 场景 |


### 4.5 面试追问

- **问**："2PC 和 TCC 怎么选？"
  - **答**：业务链路短、要强一致、能接受锁 → 2PC；链路长、要性能、资源预留明确 → TCC；能接受最终一致 → Saga/本地消息表。
- **问**："TCC 的 Confirm 失败了怎么办？"
  - **答**：重试 Confirm（必须幂等）；多次失败进人工/定时任务对账。

---


## §5 限流、熔断、降级（🔴 必背概念 + 代码）


![dist_token_bucket.png](images/dist_token_bucket.png)

*配图来源：本仓库 matplotlib 绘制（`images/dist_token_bucket.png`）*


### 5.1 限流四件套

| 算法 | 原理 | 优点 | 缺点 | 适用 |
|------|------|------|------|------|
| **固定窗口** | 每秒计数，超限拒绝 | 简单 | 窗口边界突刺（前 999ms 无请求最后 1ms 1000 个） | 粗粒度 |
| **滑动窗口** | 窗口按时间滑，计数精确 | 平滑 | 比固定窗口多存一个窗口计数 | 较精确 |
| **漏桶（Leaky Bucket）** | 请求入桶，恒定速率流出 | 输出绝对平滑 | 不能应对突发（有突发也匀速） | 平滑流出场景 |
| **令牌桶（Token Bucket）** | 恒速放令牌，请求取令牌 | **允许突发**（积攒令牌） | 实现略复杂 | 秒杀、网关（最常用） |

**令牌桶 Python 实现**（可运行）：


In [ ]:
import time
import threading

class TokenBucket:
    def __init__(self, capacity, rate):
        self.capacity = capacity        # 桶容量（最大突发）
        self.tokens = capacity          # 当前令牌数
        self.rate = rate                # 每秒补充令牌数
        self.lock = threading.Lock()
        self.last = time.time()

    def acquire(self, n=1):
        with self.lock:
            now = time.time()
            self.tokens = min(self.capacity,
                             self.tokens + (now - self.last) * self.rate)
            self.last = now
            if self.tokens >= n:
                self.tokens -= n
                return True
            return False

# 使用：QPS 上限 5，容量 3（允许突发 3 个）
bucket = TokenBucket(capacity=3, rate=5)
for _ in range(10):
    print('放行' if bucket.acquire() else '拒绝')
    time.sleep(0.1)


### 5.2 熔断（Circuit Breaker）

**状态机**（Hystrix 三态）：

```
CLOSED（正常）--失败率超阈值(N)--> OPEN（熔断，直接快速失败）
OPEN --超时冷却--> HALF_OPEN（放少量探测请求）
HALF_OPEN --探测成功--> CLOSED
HALF_OPEN --探测失败--> OPEN
```

- **熔断的目的**：下游已挂时**快速失败**，而不是让调用方继续堆积超时（防止雪崩）。
- **关键参数**：失败率阈值（如 50%）、窗口大小、熔断持续时间、半开探测请求数。


### 5.3 降级（Fallback）

- **熔断 vs 降级区别**：熔断是**系统自我保护**（下游故障触发）；降级是**主动预案**（预期到高峰，主动关掉非核心功能）。
- 例子：大促时降级"猜你喜欢"推荐、降级日志采集；返回兜底数据或缓存旧值。


### 5.4 面试追问

- **问**："限流是加在网关还是应用？"
  - **答**：**网关层**做全局粗粒度限流（IP/用户维度）+ **应用层**做业务细粒度限流（接口维度），两级配合。
- **问**："Sentinel 与 Hystrix 区别？"
  - **答**：Hystrix 基于线程池/信号量隔离，Sentinel 基于滑动窗口实时统计、支持热点限流与系统自适应，且不依赖线程池。

---


## §6 分布式 ID 与分布式锁（🔴 必背）


### 6.1 分布式 ID 方案对比

| 方案 | 特点 | 有序 | 缺点 |
|------|------|------|------|
| **UUID** | 本地生成、无中心 | 无序 | 32 位字符串、索引碎片化 |
| **数据库自增** | 简单 | 有序 | 单点、性能低 |
| **号段模式** | 一次取一段 | 有序 | 依赖 DB |
| **雪花算法** | 时间戳+机器+序列 | 趋势有序 | 依赖时钟（时钟回拨问题） |
| **Redis INCR** | 简单 | 有序 | 依赖 Redis、量大有瓶颈 |

**雪花算法 64 位布局**：

```
0 | 41bit 时间戳(ms) | 10bit 机器ID | 12bit 序列号
```

每秒可生成 `2^12 = 4096` 个/ms = 409.6 万个/秒；**时钟回拨**时暂停或拒绝生成。


### 6.2 Redis 分布式锁（SETNX + Lua，🔴 手写）

**正确姿势**（Redisson 风格）：

```lua
-- 加锁：SET key value NX EX timeout（一条命令保证原子）
-- 释放：先比对 value（防误删别人的锁）再 DEL，Lua 保证原子
if redis.call("get", KEYS[1]) == ARGV[1] then
    return redis.call("del", KEYS[1])
else
    return 0
end
```

**要点清单**：
1. **加锁**：`SET lock:key unique_id NX EX 30` —— NX 保证互斥、EX 防死锁、unique_id 防误删。
2. **释放**：必须 Lua 比较唯一标识再删（否则可能删掉别的线程刚获取的锁）。
3. **续期**：业务没跑完前用看门狗（WatchDog）每 10s 续期 30s；进程挂了自动过期。
4. **Redlock 争议**：在 5 个独立 Redis 上同时加锁，过半成功才算获锁——Martin Kleppmann 与作者争论过其是否真的安全（时钟跳跃/GC 停顿问题），一般业务用单节点 + 看门狗足够。

**Python 可运行版**：


> 🔬 **需在独立环境中运行**（依赖 redis-server，notebook 内无法连接，本 notebook 内不执行）：

```python
import uuid
import time
import redis

r = redis.Redis(host='localhost', port=6379)

def acquire_lock(name, timeout=10, ttl=30):
    token = uuid.uuid4().hex
    ok = r.set(name, token, nx=True, ex=ttl)
    if ok:
        return token
    return None

def release_lock(name, token):
    # 用 Lua 保证"比对+删除"原子
    lua = "if redis.call('get', KEYS[1]) == ARGV[1] then return redis.call('del', KEYS[1]) else return 0 end"
    return bool(r.eval(lua, 1, name, token))
```


### 6.3 ZooKeeper 分布式锁

- 创建**临时顺序节点** `/lock/lock_0001`，取最小序号者获锁；序号不是最小的则 watch 前一个节点，前一个删除后重新竞争。
- **优点**：临时节点 + 会话机制天然防死锁（客户端挂了节点自动删除）；**缺点**：性能不如 Redis、依赖 ZK 可用性。


### 6.4 面试追问

- **问**："Redis 锁过期了但业务没执行完怎么办？"
  - **答**：看门狗续期 + 释放时校验 token；若还是过期了，用 **Redisson 的锁续期 + 可重入**，极端情况接受互斥窗口（或用 ZK 语义）。
- **问**："分布式锁和本地锁什么区别？"
  - **答**：本地锁只对本进程线程有效（如 `threading.Lock`）；跨进程/跨机器必须用 Redis/ZK/DB 等外部协调器。

---


## §7 负载均衡与一致性哈希（🟡 掌握 + 代码）


![dist_consistent_hash.png](images/dist_consistent_hash.png)

*配图来源：本仓库 matplotlib 绘制（`images/dist_consistent_hash.png`）*


### 7.1 负载均衡算法对比

| 算法 | 原理 | 问题 |
|------|------|------|
| 轮询 RR | 依次分发 | 权重不敏感 |
| 加权轮询 | 按权重比例 | 热点可能偏移 |
| 最少连接 | 分给连接最少的 | 需实时统计 |
| **一致性哈希** | 哈希环 + 顺时针找节点 | 解决"节点增减只影响少量 key" |


### 7.2 一致性哈希原理与虚拟节点

**问题背景**：普通 `hash(key) % N` 在节点数变化时，几乎**所有** key 都要迁移 → 缓存雪崩。

**一致性哈希**：
1. 把节点 hash 到 0~2^32 圆环上；
2. key 也 hash，**顺时针找第一个节点**；
3. 节点增减时，**只有环上相邻区间**的 key 受影响（O(1/N) 比例迁移）。

**虚拟节点**解决"节点少导致数据倾斜"：每个物理节点复制成 160 个虚拟节点均匀撒在环上。


### 7.3 Python 实现（可运行）


In [ ]:
import hashlib
import bisect

class ConsistentHash:
    def __init__(self, nodes, replicas=160):
        self.replicas = replicas
        self.ring = []                      # (hash, node) 升序
        self.node_map = {}
        for node in nodes:
            self._add_node(node)

    def _hash(self, key):
        return int(hashlib.md5(key.encode()).hexdigest(), 16) % (2 ** 32)

    def _add_node(self, node):
        for i in range(self.replicas):
            h = self._hash(f"{node}#{i}")
            self.ring.append((h, node))
        self.ring.sort()

    def get_node(self, key):
        h = self._hash(key)
        idx = bisect.bisect_left(self.ring, (h, '')) % len(self.ring)
        return self.ring[idx][1]

# 验证：移除一个节点后大部分 key 仍落在原节点
nodes = ['A', 'B', 'C']
ch = ConsistentHash(nodes)
keys = [f'key-{i}' for i in range(1000)]
before = {k: ch.get_node(k) for k in keys}

ch2 = ConsistentHash(['A', 'B'])           # 摘掉 C
moved = sum(1 for k in keys if ch2.get_node(k) != before[k])
print(f'移除节点后 key 迁移比例: {moved / len(keys):.1%}')   # 远小于 1/3 而是接近 1/3 的环区间


### 7.4 面试追问

- **问**："一致性哈希在 Redis 集群里还用吗？"
  - **答**：Redis Cluster 用 **CRC16 % 16384 slot** 代替一致性哈希（16384 个槽位稳定、支持迁移），一致性哈希主要用于自研网关/缓存分片。
- **问**："虚拟节点太多会怎样？"
  - **答**：内存与排序开销增大；160 是工程上平衡点。

---


## §8 高频追问速答（1 分钟版）

| 问题 | 一句话答案 |
|------|-----------|
| 什么是幂等 | 同一操作执行多次结果一致（按唯一键去重/版本号） |
| 为什么 Kafka 快 | 顺序写盘 + 页缓存 + 零拷贝 + 批量 |
| 2PC 为什么慢 | Prepare 后长锁等待协调者 |
| 分布式锁怎么防误删 | 存唯一 token，删除前 Lua 比对 |
| 熔断和降级区别 | 熔断被动保护下游，降级主动牺牲非核心 |
| 最终一致怎么实现 | 异步复制/消息队列 + 幂等 + 对账 |

---


## §9 经典系统设计题速答（高频追问）


### 9.1 设计一个秒杀系统（下单链路）

1. **前端**：按钮置灰 + 本地令牌（防重复点击）
2. **接入层**：网关令牌桶限流（按 IP/用户），CDN 承载静态页
3. **削峰**：MQ 削峰，异步下单（限"预扣库存"进入队列）
4. **Redis 预扣库存**：`DECR stock` 原子扣减，<=0 直接返回"已抢光"（**防超卖**）
5. **DB 落单**：消费队列写订单 + 扣 DB 库存（带唯一订单号幂等）
6. **兜底**：超时未支付释放 Redis 库存；对账任务处理不一致

**防超卖本质**：把库存扣减收敛到**一个原子操作**（Redis DECR 或 DB `UPDATE stock SET n=n-1 WHERE n>0`）。


### 9.2 设计一个短链系统（tinyurl）

- **发号器**：雪花算法/DB 自增拿全局唯一 id → base62 编码（62^7 ≈ 3.5 万亿）→ 短码
- **存储**：MySQL（long_url 唯一索引查重）+ Redis 缓存
- **跳转**：`GET /{code}` → 查缓存 → 查库 → `302` 重定向
- **扩展**：分库分表按短码哈希；布隆过滤器拦截不存在短码


### 9.3 设计一个朋友圈 Feed 流

- **推模式（写扩散）**：发帖时写入所有粉丝的 inbox（实时但写放大，适合小 V）
- **拉模式（读扩散）**：查看时合并关注列表的最新帖（读放大，适合大 V）
- **混合**：大 V 拉模式、普通用户推模式（Twitter 的经典方案）
- **缓存**：Redis 存 timeline list（每用户 zset 按时间戳），DB 是冷备


### 9.4 设计一个限流配置中心

- 配置存 **Apollo/Nacos**（支持热更新）
- 限流规则下发到网关本地缓存（每 N 秒拉取一次 + 长轮询推送）
- 指标采集（QPS/失败率）上报到监控 → 触发**动态降级/扩容**


### 9.5 面试答题框架（无论哪题都这样讲）

1. **澄清需求**：QPS 多少？数据量多大？一致性要求？可用性要求？
2. **画架构**：客户端 → LB → 网关（限流/鉴权）→ 服务（无状态）→ 缓存 → DB/MQ
3. **逐层说细节**：数据模型、缓存怎么用、队列解决什么问题、失败怎么兜底
4. **说权衡**：CAP 取向、选型理由、瓶颈在哪、怎么扩容

> ⚠️ 高频扣分点：只画图不讲"为什么"；不提限流/缓存/异步；不说扩展性假设（单机 vs 万级 QPS 方案根本不同）。

---

**验收**：🔴 条目能 1 分钟盲讲；能手写令牌桶、Redis 分布式锁（Lua）、一致性哈希；能对比 2PC/TCC/Saga；拿到系统设计题能按"澄清→架构→细节→权衡"四步讲满 5 分钟。


## 🔴 CAP 一分钟判断卡（背下来，面试开口就有分）

### 一句话判断法

```text
数据错了会出大事（钱/库存/订单/投票）  → 选 CP，宁可拒绝请求
数据旧一会儿没关系（feed/点赞/在线状态）→ 选 AP，活着能答就行
```

### 系统归类速记

| 系统 | 取向 | 原因 |
|---|---|---|
| ZooKeeper / etcd | CP | 过半提交 + 选主，leader 失联拒绝写 |
| Eureka | AP | 服务发现宁可给旧列表也不宕 |
| Nacos | 默认 AP 可切 CP | 双模型 |
| Redis Cluster | AP | 主从异步，主挂可能丢写 |
| MySQL 主从 | 最终一致 | binlog 异步复制 |

### 一句话论证"为什么三选二"

分区发生时，节点间无法通信：想让两边都保持"可用"，就必然出现一边数据旧 → 二选一。分区是常态不是假设，所以**必须保证 P**，实际选择只有 CP vs AP。

---


In [ ]:
# hash_migrate.py —— 普通取模 vs 一致性哈希：节点 3→2 时 key 迁移比例对比
import hashlib
import bisect

def mod_hash(key, n):
    h = int(hashlib.md5(key.encode()).hexdigest(), 16)
    return h % n

class ConsistentHash:
    def __init__(self, nodes, replicas=160):
        self.replicas = replicas
        self.ring = []
        for node in nodes:
            for i in range(replicas):
                h = int(hashlib.md5(f'{node}#{i}'.encode()).hexdigest(), 16)
                self.ring.append((h, node))
        self.ring.sort()

    def get_node(self, key):
        h = int(hashlib.md5(key.encode()).hexdigest(), 16)
        idx = bisect.bisect_left(self.ring, (h, '')) % len(self.ring)
        return self.ring[idx][1]

keys = [f'key-{i}' for i in range(1000)]

# 普通取模: hash % 3 → hash % 2
mod_moved = sum(1 for k in keys if mod_hash(k, 3) != mod_hash(k, 2))
print(f'普通取模 3→2: 迁移 {mod_moved / len(keys):.1%} 的 key   ← 几乎全迁, 缓存雪崩')

# 一致性哈希: 摘掉节点 C
ch3 = ConsistentHash(['A', 'B', 'C'])
ch2 = ConsistentHash(['A', 'B'])
before = {k: ch3.get_node(k) for k in keys}
ch_moved = sum(1 for k in keys if ch2.get_node(k) != before[k])
print(f'一致性哈希 3→2: 仅迁移 {ch_moved / len(keys):.1%} 的 key   ← 只影响被摘节点环区间')
print('结论: 取模适合节点稳定的场景; 缓存/网关分片必须一致性哈希 + 虚拟节点')


## 🟡 分布式事务方案决策树（面试画图讲）

```text
业务链路需要强一致? ──是──→ 链路短+可接受锁? ──是──→ 2PC / 3PC
     │                     └─否──→ 单库事务/本地事务拆分
     └─否（可最终一致）──→ 对账容忍度?
              ├─ 高 → Saga / 本地消息表
              └─ 低 → 事务消息(RocketMQ) / TCC
```

### 一句话区别（背）

- **2PC**：强一致、同步阻塞、协调者单点、长锁低性能
- **TCC**：业务幂等预留（Try/Confirm/Cancel），性能好，侵入高（三套代码）
- **Saga**：正向操作 + 补偿操作，最终一致，长流程首选
- **本地消息表**：业务与消息同库同事务，后台发 MQ——生产端不丢消息最简方案

### 追问必答

Q: TCC 的 Confirm 失败？→ 幂等重试；多次失败 → 定时对账/人工介入。
Q: 2PC 协调者挂了？→ 用 Raft/Paxos 替换协调者（如 MySQL Group Replication）或恢复后人工补偿。

---


In [ ]:
# two_phase_commit.py —— 2PC 协调者状态机模拟（内存版：全部同意→提交 / 任一反对→回滚）
class Node:
    def __init__(self, name, will_commit=True):
        self.name, self.will_commit = name, will_commit
        self.log = []

    def prepare(self):
        ok = self.will_commit
        self.log.append(f'prepare→{ok} (写 undo/redo 日志但不提交, 资源加锁)')
        return ok

    def commit(self):
        self.log.append('commit (真正落盘, 释放锁)')
        return 'OK'

    def rollback(self):
        self.log.append('rollback (按 undo 日志回滚, 释放锁)')
        return 'OK'


class Coordinator:
    def __init__(self, participants):
        self.participants = participants

    def run(self):
        # 阶段一: Prepare（问所有人能不能提交, 谁都不许提交）
        votes = {n: node.prepare() for n, node in self.participants.items()}
        # 阶段二: 全票通过→Commit, 否则→Rollback
        if all(votes.values()):
            state = 'COMMIT'
            outcome = {n: node.commit() for n, node in self.participants.items()}
        else:
            state = 'ROLLBACK'
            outcome = {n: node.rollback() for n, node in self.participants.items()}
        return state, votes, outcome


ok = {'库存服务': Node('库存'), '订单服务': Node('订单')}
bad = {'库存服务': Node('库存'), '订单服务': Node('订单', will_commit=False)}
for name, parts in [('全部同意', ok), ('订单服务反对', bad)]:
    state, votes, outcome = Coordinator(parts).run()
    print(f'[{name}] 投票={votes} → 最终状态={state}')
    for node in parts.values():
        print('  ', node.log[-2])
        print('  ', node.log[-1])


## 🔴 限流四件套背诵卡

| 算法 | 一句话原理 | 突发处理 | 适用 |
|---|---|---|---|
| 固定窗口 | 每秒计数超限拒绝 | 窗口边界突刺 | 粗粒度限流 |
| 滑动窗口 | 精确到秒内任意长度窗口 | 平滑 | 网关精确限流 |
| 漏桶 | 恒定速率流出 | 不处理突发（匀速） | 输出平滑场景 |
| 令牌桶 | 恒速放令牌、请求取令牌 | **允许突发** | 秒杀/网关默认选择 |

**选型一句话**：要"平滑输出"选漏桶；要"允许突发又能兜住平均速率"选令牌桶。
**位置**：网关层做 IP/用户维度粗粒度限流 + 应用层做接口维度细粒度限流。

---


In [ ]:
# sliding_window.py —— 滑动窗口限流器（deque 存时间戳, 过期就裁掉）
import time
from collections import deque

class SlidingWindow:
    def __init__(self, limit, window=1.0):
        self.limit = limit
        self.window = window
        self.ts = deque()

    def allow(self):
        now = time.time()
        while self.ts and now - self.ts[0] >= self.window:
            self.ts.popleft()          # 滑出窗口的请求作废
        if len(self.ts) < self.limit:
            self.ts.append(now)
            return True
        return False


sw = SlidingWindow(limit=3, window=1.0)
for i in range(10):
    print(f'第{i + 1:02d} 个请求:', '放行' if sw.allow() else '❌ 拒绝')
    time.sleep(0.05)
print('注: 与固定窗口相比, 滑动窗口不会出现"窗口边缘突刺"——任意 1s 内都最多 3 个')


In [ ]:
# circuit_breaker.py —— 熔断器三态状态机（Hystrix 风格简化版）
import random
import time

class CircuitBreaker:
    def __init__(self, fail_threshold=2, cooldown=1):
        self.fail_threshold = fail_threshold
        self.cooldown = cooldown
        self.state = 'CLOSED'     # CLOSED 正常 / OPEN 熔断 / HALF_OPEN 半开探测
        self.fails = 0
        self.open_at = None

    def call(self, fn):
        now = time.time()
        if self.state == 'OPEN':
            if now - self.open_at >= self.cooldown:
                self.state = 'HALF_OPEN'   # 冷却结束, 放少量探测请求
            else:
                return '❌ 熔断快速失败(不调用下游)'
        try:
            r = fn()
        except RuntimeError:
            self.fails += 1
            if self.state == 'HALF_OPEN' or self.fails >= self.fail_threshold:
                self.state = 'OPEN'
                self.open_at = now
                self.fails = 0
            return '❌ 调用失败'
        self.fails = 0
        if self.state == 'HALF_OPEN':
            self.state = 'CLOSED'         # 探测成功 → 恢复正常
        return r


def ok():
    return '✅ 成功'

def bad():
    raise RuntimeError('下游超时')

random.seed(7)   # 固定失败序列, 结果可复现
cb = CircuitBreaker(fail_threshold=2, cooldown=1)
for i in range(15):
    fn = ok if random.random() > 0.55 else bad
    print(f'{i:02d} 状态={cb.state:>10}  {cb.call(fn)}')
    time.sleep(0.15)
print('观察: 连续失败→OPEN 快速失败; 冷却后 HALF_OPEN 探测; 成功→CLOSED 恢复')


In [ ]:
# snowflake.py —— 雪花算法 64 位 ID 生成（41 位时间戳 | 10 位机器 | 12 位序列）
import time

class Snowflake:
    def __init__(self, worker_id=1, datacenter_id=1):
        self.worker = worker_id & 0x3FF        # 低 10 位
        self.datacenter = datacenter_id & 0x1F # 低 5 位
        self.seq = 0
        self.last_ms = -1

    def next_id(self):
        ms = int(time.time() * 1000)
        if ms == self.last_ms:
            self.seq = (self.seq + 1) & 0xFFF   # 同毫秒内序列自增, 12 位
            if self.seq == 0:                   # 这一毫秒的 4096 个用完
                while ms <= self.last_ms:        # 阻塞等下一毫秒
                    ms = int(time.time() * 1000)
        else:
            self.seq = 0
        self.last_ms = ms
        return (ms << 22) | (self.datacenter << 17) | (self.worker << 12) | self.seq


sf = Snowflake(worker_id=3)
ids = [sf.next_id() for _ in range(5)]
for i, v in enumerate(ids):
    print(f'ID{i}: {v}  二进制 {v.bit_length()} 位')
print('单调递增:', all(b > a for a, b in zip(ids, ids[1:])))
print('注意: 跨机器单调依赖时钟; 时钟回拨会重复 → 回拨时暂停生成或拒绝(业界按回拨量处理)')


## 🔴 分布式 ID 方案背诵卡 + 雪花算法考点

| 方案 | 有序? | 优点 | 缺点 |
|---|---|---|---|
| UUID | 无序 | 本地生成无中心 | 长字符串、索引碎片化、无业务含义 |
| DB 自增 | 有序 | 简单 | 单点、性能低 |
| 号段模式 | 有序 | 一次取一段缓存内存发号 | 依赖 DB、段用尽要取新段 |
| 雪花算法 | 趋势有序 | 无中心、高性能(4096/ms/机) | 依赖时钟(回拨问题) |
| Redis INCR | 有序 | 简单 | 依赖 Redis、量大有瓶颈 |

**雪花 64 位布局**：`0 | 41bit 毫秒时间戳 | 10bit 机器ID | 12bit 序列号`，单机每秒 409.6 万个。
**时钟回拨对策**：① 回拨 <5ms 时等待追平；② 回拨较大时用备用 ID 段/暂停发号；③ 变体（如百度 uid-generator 双号段）切换段。

---


In [ ]:
# idempotent.py —— 幂等消费三招之一：唯一键去重表（MQ 至少一次投递下的兜底）
class DedupTable:
    """模拟 DB 唯一索引: 重复 order_id 的 INSERT 会失败"""
    def __init__(self):
        self.seen = set()

    def try_consume(self, order_id, handler):
        if order_id in self.seen:
            return f'重复消息 {order_id} → 跳过(已处理过)'
        self.seen.add(order_id)      # 先落唯一键(事务里等同 INSERT IGNORE)
        return handler(order_id)


def deduct_stock(oid):
    return f'订单 {oid}: 扣减库存成功'


dt = DedupTable()
for oid in ['O1', 'O1', 'O2', 'O1']:   # 至少一次投递 → O1 被消费 3 次
    print(dt.try_consume(oid, deduct_stock))
print('另两招: 版本号乐观锁(UPDATE ... WHERE version=? 影响 0 行即重复) / Redis SETNX')
print('面试答"重复消费怎么办": 幂等三招背完 → 补一句"配合消息表对账兜底"')


## 🟡 面试追问 10 连问速答（分布式方向）

1. **为什么 Kafka 快**：顺序写盘 + 页缓存 + 零拷贝(sendfile) + 批量压缩
2. **消息积压怎么办**：先加消费者(超过分区数需先加分区) → 定位消费慢还是生产快 → 紧急可临时转储
3. **怎么保证不丢消息**：生产端 send 同步 ack + 重试/本地消息表；Broker acks=all + ISR；消费端先业务后提交 offset
4. **怎么保证顺序**：同一 key 哈希进同一分区 + 单线程消费
5. **Redis 锁过期业务没跑完**：看门狗续期 + 释放校验 token（Redisson）
6. **Redlock 争议**：时钟跳跃/GC 停顿下五节点过半也可能失效；普通业务单节点锁 + 续期足够
7. **熔断 vs 降级**：熔断=下游故障被动保护；降级=大促主动牺牲非核心
8. **最终一致怎么做**：异步复制/MQ + 幂等 + 定期对账
9. **一致性哈希还用吗**：Redis Cluster 用 CRC16%16384 槽位；一致性哈希用于自研分片/网关
10. **限流放网关还是应用**：两级配合——网关粗粒度 + 应用细粒度

---


## 🔴 系统设计题四步答题框架（任何题目套用）

```text
① 澄清需求: QPS? 数据量? 一致性? 可用性? 读多还是写多?
② 画架构: 客户端 → LB → 网关(限流/鉴权) → 服务(无状态) → 缓存 → DB/MQ
③ 逐层细节: 数据模型 / 缓存策略 / 队列用途 / 失败兜底
④ 说权衡: CAP 取向 / 选型理由 / 瓶颈点 / 扩容方案
```

**高频扣分点**（背下来避雷）：
- 只画图不讲"为什么这么做"
- 不提限流 / 缓存 / 异步这三个保命词
- 不说清单机 vs 万级 QPS 方案的不同
- 没有"如果 X 挂了怎么办"的兜底讨论

**秒杀防超卖本质**：把库存扣减收敛到一个原子操作（Redis `DECR` 或 DB `UPDATE stock=n-1 WHERE n>0`），加唯一订单号幂等。
